# 01. Raw Data Inspection & Quality Assessment
**Phase 1: Project Foundation, Data Collection, Data Dictionary and Privacy**
**Project:** Hospital Patient Data Analysis and Dashboard Development Using Python and Power BI

---

### Objective & Privacy Protocol
This notebook performs an exhaustive preliminary inspection of the raw healthcare dataset (`data/raw/healthcare_dataset.csv`), evaluating structure, schema fidelity, completeness, cardinality, date bounds, and value distributions.

> **CRITICAL PRIVACY DIRECTIVE**:
> In compliance with healthcare data protection standards and project non-negotiables, all direct personal identifiers (`Name`, `Doctor`) are **strictly masked** in all tabular previews and console outputs to prevent privacy leaks.


In [1]:
# Environment configuration, seed setting, and package imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for complete reproducibility
np.random.seed(42)

# Set display preferences
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print(f"Python interpreter: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


C:\Users\berab\.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at C:\Users\berab\AppData\Local\Temp\matplotlib-zftsyvs1 because there was an issue with the default path (C:\Users\berab\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Python interpreter: 3.11.9
Pandas version: 2.2.1
NumPy version: 1.26.4


## 1. Raw Dataset Ingestion & Dimension Verification
Loading raw data from `data/raw/healthcare_dataset.csv` using relative paths.

In [2]:
# Resolve relative path whether running from root or notebooks/
raw_path = 'data/raw/healthcare_dataset.csv'
if not os.path.exists(raw_path):
    raw_path = '../data/raw/healthcare_dataset.csv'

assert os.path.exists(raw_path), f"File not found at {raw_path}"

df_raw = pd.read_csv(raw_path)
n_rows, n_cols = df_raw.shape

print(f"Loaded dataset successfully from: {raw_path}")
print(f"Total Admissions (Rows) : {n_rows:,}")
print(f"Total Columns (Fields)  : {n_cols}")


Loaded dataset successfully from: data/raw/healthcare_dataset.csv
Total Admissions (Rows) : 55,500
Total Columns (Fields)  : 15


## 2. Privacy-Safe Tabular Preview
Previewing sample records with `Name` and `Doctor` completely masked.

In [3]:
def privacy_safe_preview(df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """Return a copy of the dataframe with direct identifiers masked."""
    preview = df.head(n).copy()
    if 'Name' in preview.columns:
        preview['Name'] = [f"PATIENT_{str(i+1).zfill(5)} [MASKED]" for i in range(len(preview))]
    if 'Doctor' in preview.columns:
        preview['Doctor'] = [f"DOCTOR_{str(i+1).zfill(3)} [MASKED]" for i in range(len(preview))]
    return preview

preview_df = privacy_safe_preview(df_raw, 5)
print(preview_df.to_string())


                     Name  Age  Gender Blood Type Medical Condition Date of Admission               Doctor                    Hospital Insurance Provider  Billing Amount  Room Number Admission Type Discharge Date   Medication  Test Results
0  PATIENT_00001 [MASKED]   30    Male         B-            Cancer        2024-01-31  DOCTOR_001 [MASKED]             Sons and Miller         Blue Cross       18,856.28          328         Urgent     2024-02-02  Paracetamol        Normal
1  PATIENT_00002 [MASKED]   62    Male         A+           Obesity        2019-08-20  DOCTOR_002 [MASKED]                     Kim Inc           Medicare       33,643.33          265      Emergency     2019-08-26    Ibuprofen  Inconclusive
2  PATIENT_00003 [MASKED]   76  Female         A-           Obesity        2022-09-22  DOCTOR_003 [MASKED]                    Cook PLC              Aetna       27,955.10          205      Emergency     2022-10-07      Aspirin        Normal
3  PATIENT_00004 [MASKED]   28  Female  

## 3. Schema, Data Types & Missing Value Audit

In [4]:
# Inspect data types and missing counts across all columns
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw)) * 100

schema_audit_df = pd.DataFrame({
    'Column Name': df_raw.columns,
    'Non-Null Count': df_raw.notnull().sum(),
    'Null Count': null_counts,
    'Null Percentage (%)': null_pct,
    'Data Type': df_raw.dtypes.astype(str)
}).reset_index(drop=True)

print("Comprehensive Schema & Missing Value Audit:")
print(schema_audit_df.to_string())


Comprehensive Schema & Missing Value Audit:
           Column Name  Non-Null Count  Null Count  Null Percentage (%) Data Type
0                 Name           55500           0                 0.00    object
1                  Age           55500           0                 0.00     int64
2               Gender           55500           0                 0.00    object
3           Blood Type           55500           0                 0.00    object
4    Medical Condition           55500           0                 0.00    object
5    Date of Admission           55500           0                 0.00    object
6               Doctor           55500           0                 0.00    object
7             Hospital           55500           0                 0.00    object
8   Insurance Provider           55500           0                 0.00    object
9       Billing Amount           55500           0                 0.00   float64
10         Room Number           55500           0    

## 4. Cardinality & Unique Value Analysis

In [5]:
# Assess distinct counts per field
unique_counts = df_raw.nunique()
cardinality_df = pd.DataFrame({
    'Column Name': df_raw.columns,
    'Distinct Count': unique_counts,
    'Uniqueness Ratio (%)': ((unique_counts / len(df_raw)) * 100).round(2),
    'Classification': [
        'Direct Identifier (High Cardinality)' if col in ['Name', 'Doctor']
        else 'Non-Identifying High Cardinality' if u > 1000
        else 'Quasi-Identifier / Numeric' if u > 10
        else 'Categorical Domain'
        for col, u in zip(df_raw.columns, unique_counts)
    ]
}).reset_index(drop=True)

print("Field Cardinality & Unique Value Distribution:")
print(cardinality_df.to_string())


Field Cardinality & Unique Value Distribution:
           Column Name  Distinct Count  Uniqueness Ratio (%)                        Classification
0                 Name           49992                 90.08  Direct Identifier (High Cardinality)
1                  Age              77                  0.14            Quasi-Identifier / Numeric
2               Gender               2                  0.00                    Categorical Domain
3           Blood Type               8                  0.01                    Categorical Domain
4    Medical Condition               6                  0.01                    Categorical Domain
5    Date of Admission            1827                  3.29      Non-Identifying High Cardinality
6               Doctor           40341                 72.69  Direct Identifier (High Cardinality)
7             Hospital           39876                 71.85      Non-Identifying High Cardinality
8   Insurance Provider               5                  0.01  

## 5. Temporal Boundaries & Stay Duration Analysis
Assessing `Date of Admission` and `Discharge Date` chronology and stay integrity.

In [6]:
# Parse datetime fields
adm_dates = pd.to_datetime(df_raw['Date of Admission'])
dis_dates = pd.to_datetime(df_raw['Discharge Date'])

stay_duration_days = (dis_dates - adm_dates).dt.days

print("=" * 60)
print("TEMPORAL BOUNDS AUDIT")
print("=" * 60)
print(f"Earliest Admission Date : {adm_dates.min().strftime('%Y-%m-%d')}")
print(f"Latest Admission Date   : {adm_dates.max().strftime('%Y-%m-%d')}")
print(f"Total Admission Span    : {(adm_dates.max() - adm_dates.min()).days} calendar days")
print(f"Earliest Discharge Date : {dis_dates.min().strftime('%Y-%m-%d')}")
print(f"Latest Discharge Date   : {dis_dates.max().strftime('%Y-%m-%d')}")

# Chronological validation
inverted_stays = (stay_duration_days < 0).sum()
same_day_stays = (stay_duration_days == 0).sum()
print(f"Negative Stay Anomalies (Discharge < Admission) : {inverted_stays}")
print(f"Same-Day Stays (Discharge == Admission)          : {same_day_stays}")
print("\nStay Duration Distribution (Raw Days):")
print(stay_duration_days.describe().to_string())


TEMPORAL BOUNDS AUDIT
Earliest Admission Date : 2019-05-08
Latest Admission Date   : 2024-05-07
Total Admission Span    : 1826 calendar days
Earliest Discharge Date : 2019-05-09
Latest Discharge Date   : 2024-06-06
Negative Stay Anomalies (Discharge < Admission) : 0
Same-Day Stays (Discharge == Admission)          : 0

Stay Duration Distribution (Raw Days):
count   55,500.00
mean        15.51
std          8.66
min          1.00
25%          8.00
50%         15.00
75%         23.00
max         30.00


## 6. Numeric Field Summaries & Anomaly Detection
Auditing `Age`, `Billing Amount`, and `Room Number` distributions.

In [7]:
numeric_cols = ['Age', 'Billing Amount', 'Room Number']
num_stats = df_raw[numeric_cols].describe().T
num_stats['median'] = df_raw[numeric_cols].median()
num_stats['skewness'] = df_raw[numeric_cols].skew()
num_stats['kurtosis'] = df_raw[numeric_cols].kurtosis()

print("Descriptive Statistics for Numeric Columns:")
print(num_stats[['count', 'mean', 'std', 'min', '25%', 'median', '75%', 'max', 'skewness', 'kurtosis']].to_string())

# Deep dive into anomalies: negative billing amounts
neg_billing = df_raw[df_raw['Billing Amount'] < 0]
print(f"\nNegative Billing Amount Records Detected: {len(neg_billing)}")
if len(neg_billing) > 0:
    print(f"Minimum Billing Amount: ${df_raw['Billing Amount'].min():,.2f}")
    print(f"Maximum Billing Amount: ${df_raw['Billing Amount'].max():,.2f}")


Descriptive Statistics for Numeric Columns:
                   count      mean       std       min       25%    median       75%       max  skewness  kurtosis
Age            55,500.00     51.54     19.60     13.00     35.00     52.00     68.00     89.00     -0.01     -1.19
Billing Amount 55,500.00 25,539.32 14,211.45 -2,008.49 13,241.22 25,538.07 37,820.51 52,764.28     -0.00     -1.19
Room Number    55,500.00    301.13    115.24    101.00    202.00    302.00    401.00    500.00     -0.01     -1.19

Negative Billing Amount Records Detected: 108
Minimum Billing Amount: $-2,008.49
Maximum Billing Amount: $52,764.28


## 7. Categorical Field Distributions & Value Counts

In [8]:
cat_columns = [
    'Gender', 'Blood Type', 'Medical Condition', 
    'Insurance Provider', 'Admission Type', 'Medication', 'Test Results'
]

for col in cat_columns:
    counts = df_raw[col].value_counts()
    pcts = df_raw[col].value_counts(normalize=True) * 100
    summary_cat = pd.DataFrame({'Frequency': counts, 'Percentage (%)': pcts.round(2)})
    print(f"\n{'='*20} {col.upper()} {'='*20}")
    print(summary_cat.to_string())



==================== GENDER ====================
        Frequency  Percentage (%)
Gender                           
Male        27774           50.04
Female      27726           49.96

==================== BLOOD TYPE ====================
            Frequency  Percentage (%)
Blood Type                           
A-               6969           12.56
A+               6956           12.53
AB+              6947           12.52
AB-              6945           12.51
B+               6945           12.51
B-               6944           12.51
O+               6917           12.46
O-               6877           12.39

==================== MEDICAL CONDITION ====================
                   Frequency  Percentage (%)
Medical Condition                           
Arthritis               9308           16.77
Diabetes                9304           16.76
Hypertension            9245           16.66
Obesity                 9231           16.63
Cancer                  9227           16.63
Asth

## 8. Summary of Data Inspection Findings & Next Steps

### Key Diagnostic Observations:
1. **Dimensions**: 55,500 admissions across 15 source columns.
2. **Data Completeness**: Zero null or missing values across all columns.
3. **Temporal Span**: Admissions span 5 complete calendar years (2019-05-08 to 2024-05-07).
4. **Stay Durations**: Mean stay duration is ~15.5 days (range: 1 to 30 days). No negative stay anomalies detected.
5. **Billing Artifact**: Billed amounts range from -$2,008.49 to +$52,764.28 with a median of $25,538.07. Outlier handling / validation will be addressed in Phase 2.
6. **Demographics**: Patient age spans 13 to 89 years (mean: 51.5 years). Gender is evenly split (50.0% Male, 50.0% Female).
7. **Privacy Requirements**: `Name` and `Doctor` are high-cardinality direct identifiers that must be transformed into `patient_key` and `doctor_id` in downstream processing.
